# 3. Витрина признаков

Одна строка на клиента эксперимента. Признаки считаются SQL-запросом `features` из `sql/features.sql` только
по purchases до `CUTOFF` включительно (2019-03-18) и по `clients` с датами, сдвинутыми на 3 часа (этап 1).
`first_redeem_date` после `CUTOFF` не используется ни в каком виде: от него остаются только `redeemed_before_cutoff`
и `days_issue_to_redeem`, которые смотрят на даты не позже `CUTOFF`.

Признаки строятся только из данных до `CUTOFF`: это гарантирует фильтр по дате в запросе, ниже он проверен
пересчётом шести признаков из сырых строк purchases. Признаки не смотрят ни на target, ни на treatment: нет
кодирования по target и других статистик, которые считаются по ответам. Поэтому витрину можно строить сразу на всех
200 039 клиентах, и утечки между train, valid и test нет: признак клиента считается только по его собственной
истории.

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config, db, features, stats

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 200)
con = db.connect()

## Признаки

Окна - последние 7, 14 и 30 дней до `CUTOFF` включительно и вся история (с 2018-11-21, около 4 месяцев).
Агрегаты по чекам считаются через CTE «одна строка на чек» с ключом (client_id, transaction_id): сумма чека
и баллы берутся один раз на чек. Число товаров - по строкам чеков.

In [2]:
GROUPS = [
    ("n_trn_{w}", "число чеков"),
    ("sum_{w}", "сумма покупок, руб. (purchase_sum)"),
    ("avg_check_{w}", "средний чек, руб.; NULL, если чеков в окне нет"),
    ("reg_received_{w}", "начислено обычных баллов"),
    ("reg_spent_{w}", "списано обычных баллов"),
    ("exp_spent_{w}", "списано экспресс-баллов"),
    ("share_spend_{w}", "доля чеков со списанием баллов; NULL, если чеков в окне нет"),
    ("n_stores_{w}", "число разных магазинов"),
    ("n_products_{w}", "число разных товаров"),
]
rows = [{"признак": name.format(w="7d / 14d / 30d / all"), "что это": desc} for name, desc in GROUPS]
rows += [
    {"признак": "exp_received_all", "что это": "начислено экспресс-баллов за всю историю (только all, см. ниже)"},
    {"признак": "days_since_last", "что это": "дней от последней покупки до CUTOFF"},
    {"признак": "days_since_first", "что это": "дней от первой покупки в истории до CUTOFF"},
    {"признак": "trend_trn_14d", "что это": "чеки: (последние 14 дней - 14 дней до них) / их сумма, от -1 до 1"},
    {"признак": "trend_sum_14d", "что это": "то же для суммы покупок"},
    {"признак": "age", "что это": "возраст; NULL вне 14-100"},
    {"признак": "gender", "что это": "пол: F, M, U"},
    {"признак": "card_age_days", "что это": "давность карты на CUTOFF, дней"},
    {"признак": "redeemed_before_cutoff", "что это": "списывал ли баллы до CUTOFF включительно"},
    {"признак": "days_issue_to_redeem", "что это": "дней от выдачи карты до первого списания; NULL, если не списывал до CUTOFF или списание раньше выдачи"},
]
pd.DataFrame(rows)

,признак,что это
0,n_trn_7d / 14d / 30d / all,число чеков
1,sum_7d / 14d / 30d / all,"сумма покупок, руб. (purchase_sum)"
2,avg_check_7d / 14d / 30d / all,"средний чек, руб.; NULL, если чеков в окне нет"
3,reg_received_7d / 14d / 30d / all,начислено обычных баллов
4,reg_spent_7d / 14d / 30d / all,списано обычных баллов
5,exp_spent_7d / 14d / 30d / all,списано экспресс-баллов
6,share_spend_7d / 14d / 30d / all,"доля чеков со списанием баллов; NULL, если чек..."
7,n_stores_7d / 14d / 30d / all,число разных магазинов
8,n_products_7d / 14d / 30d / all,число разных товаров
9,exp_received_all,начислено экспресс-баллов за всю историю (толь...


9 групп по 4 окна - 36 признаков, плюс 10 отдельных: всего 46.

Правило для почти постоянных признаков: если ненулевое значение у меньше 0,5% клиентов, признак убираю. Такая
колонка почти всё время одна и та же, бустинг редко на ней что-то разделит, а если разделит - по горстке клиентов.
Под правило попали только начисления экспресс-баллов за 7, 14 и 30 дней: экспресс-баллы начисляются редкими акциями,
последняя - 2019-03-10. Проверка по всем окнам экспресс-баллов, считается прямо из purchases:

In [3]:
db.run(con, "features.sql", "express_windows").T.round(4).rename(columns={0: "доля клиентов с ненулевой суммой"})

,доля клиентов с ненулевой суммой
received_7d,0.0000
received_14d,0.0004
received_30d,0.0006
received_all,0.0186
spent_7d,0.0149
spent_14d,0.0340
spent_30d,0.0692
spent_all,0.1812


Начисления за 7, 14 и 30 дней ненулевые у 0%, 0,04% и 0,06% клиентов - убраны, за всю историю - у 1,86%, оставлены.
Списания экспресс-баллов за 7 и 14 дней ненулевые у 1,49% и 3,40% - выше порога, оставлены.

## Сборка

In [4]:
start = time.time()
features.build()
print(f"витрина собрана за {time.time() - start:.0f} с")

витрина собрана за 15 с


## Проверки витрины

In [5]:
db.run(con, "features.sql", "mart_checks")

,n_rows,n_clients,n_experiment,not_in_experiment,n_columns
0,200039,200039,200039,0,47


200 039 строк - ровно клиенты эксперимента, `client_id` уникален, лишних клиентов нет. 47 колонок: `client_id`
и 46 признаков.

In [6]:
db.run(con, "features.sql", "mart_gender")

,gender,n_clients
0,F,73696
1,M,33511
2,U,92832


Почти постоянные признаки во всей витрине: доля ненулевых значений и доля самого частого значения (NULL считается
значением).

In [7]:
db.run(con, "features.sql", "mart_constancy").round(4)

,feature,nonzero_share,top_value_share
0,exp_spent_7d,0.0149,0.9851
1,exp_received_all,0.0186,0.9814
2,exp_spent_14d,0.0340,0.9660
3,exp_spent_30d,0.0692,0.9308
4,reg_spent_7d,0.0733,0.9267
5,share_spend_7d,0.0747,0.5481
6,reg_spent_14d,0.1510,0.8490
7,share_spend_14d,0.1541,0.7129
8,exp_spent_all,0.1812,0.8188
9,reg_spent_30d,0.2817,0.7183


Меньше всего ненулевых у `exp_spent_7d` - 1,49%, это выше порога 0,5%. Ниже порога в витрине ничего нет. У пола
самое частое значение - `U`, 46,4%.

In [8]:
profile = db.run(con, "features.sql", "mart_profile").set_index("feature")
order = [c for c in pd.read_parquet(config.FEATURES_PARQUET).columns if c in profile.index]
profile.loc[order].round(3)

,null_share,min,median,max
feature,,,,
age,0.004,14.00,45.000,100.00
card_age_days,0.000,3.00,488.000,713.00
redeemed_before_cutoff,0.000,0.00,1.000,1.00
days_issue_to_redeem,0.201,0.00,121.000,702.00
n_trn_7d,0.000,0.00,1.000,44.00
n_trn_14d,0.000,0.00,2.000,69.00
n_trn_30d,0.000,1.00,4.000,119.00
n_trn_all,0.000,1.00,15.000,320.00
sum_7d,0.000,0.00,258.000,41036.90


In [9]:
mart = pd.read_parquet(config.FEATURES_PARQUET)
print("days_issue_to_redeem < 0:", int((mart.days_issue_to_redeem < 0).sum()))
print("days_issue_to_redeem NULL:", round(mart.days_issue_to_redeem.isna().mean(), 4))
print("клиентов без чеков в окне 7 дней:", int((mart.n_trn_7d == 0).sum()), "14 дней:", int((mart.n_trn_14d == 0).sum()))

days_issue_to_redeem < 0: 0
days_issue_to_redeem NULL: 0.2007
клиентов без чеков в окне 7 дней: 75469 14 дней: 26597


- NULL только там, где значение не определено: средний чек и доля чеков со списанием - когда в окне нет чеков
  (37,7% клиентов за 7 дней, 13,3% за 14 дней); `days_issue_to_redeem` - у 20,1%: кто не списывал баллы до `CUTOFF`,
  и один клиент, у которого первое списание на 43 дня раньше выдачи карты (аномалия из этапа 1, заменил на NULL);
  возраст - у 0,4%.
- За 30 дней и за всю историю чек есть у всех: все клиенты покупали хотя бы раз в окне с 2019-02-24 по 2019-03-18 (этап 1). Поэтому
  у трендов NULL нет.
- Диапазоны правдоподобные: давность последней покупки 0-22 дня, история до 117 дней, давность карты 3-713 дней.
- Суммы и число чеков с длинным хвостом (сумма за всю историю до 504 490 руб.). Для бустинга масштаб не важен,
  логарифмы не беру.

## Независимая сверка

Пересчитываю шесть признаков для 300 случайных клиентов в pandas прямо из строк purchases и сравниваю с витриной.
Так проверяются ключ чека и границы окон.

In [10]:
sample = con.execute(
    f"select client_id from uplift_train using sample reservoir(300 rows) repeatable ({config.SEED})"
).df()
con.register("sample_ids", sample)
lines = con.execute("""
    select client_id, transaction_id, transaction_datetime, purchase_sum, regular_points_spent, product_id
    from purchases
    where client_id in (select client_id from sample_ids)
        and transaction_datetime::date <= getvariable('cutoff')
""").df()
cutoff = pd.Timestamp(config.CUTOFF)
lines["age"] = (cutoff - lines.transaction_datetime.dt.normalize()).dt.days
checks = lines.drop_duplicates(["client_id", "transaction_id"])
by = checks.groupby("client_id")
last14 = checks[checks.age < 14].groupby("client_id").size()
prev14 = checks[(checks.age >= 14) & (checks.age < 28)].groupby("client_id").size()
ids = sample.client_id
a, b = last14.reindex(ids, fill_value=0), prev14.reindex(ids, fill_value=0)
manual = pd.DataFrame({
    "n_trn_7d": checks[checks.age < 7].groupby("client_id").size().reindex(ids, fill_value=0),
    "sum_30d": checks[checks.age < 30].groupby("client_id").purchase_sum.sum().reindex(ids, fill_value=0),
    "n_products_14d": lines[lines.age < 14].groupby("client_id").product_id.nunique().reindex(ids, fill_value=0),
    "days_since_last": by.age.min().reindex(ids),
    "reg_spent_all": -by.regular_points_spent.sum().reindex(ids),
    "trend_trn_14d": ((a - b) / (a + b)).reindex(ids),
})
from_mart = mart.set_index("client_id").loc[ids, manual.columns]
pd.Series({col: np.allclose(manual[col].to_numpy(float), from_mart[col].to_numpy(float), equal_nan=True) for col in manual},
          name="совпадает")

n_trn_7d           True
sum_30d            True
n_products_14d     True
days_since_last    True
reg_spent_all      True
trend_trn_14d      True
Name: совпадает, dtype: bool

Все шесть признаков совпадают: фильтр по дате и ключ чека работают так, как задумано.

## Баланс признаков между группами

Признаки до коммуникации должны быть сбалансированы между treatment и control, как ковариаты на этапе 2. Treatment
здесь используется только для этой проверки, в витрину он не входит.

In [11]:
moments = db.run(con, "features.sql", "mart_moments")
moments["smd"] = stats.smd(moments.mean_treatment, moments.mean_control, moments.var_treatment, moments.var_control)
print(f"числовых признаков: {len(moments)}, max |SMD| = {moments.smd.abs().max():.4f}")
moments.reindex(moments.smd.abs().sort_values(ascending=False).index).head(5).round(4)

числовых признаков: 45, max |SMD| = 0.0087


,feature,mean_treatment,mean_control,var_treatment,var_control,smd
17,n_stores_7d,0.8042,0.7974,0.6012,0.5907,0.0087
9,days_since_first,97.3852,97.1526,721.7189,735.2938,0.0086
18,trend_trn_14d,0.0216,0.0267,0.3592,0.3604,-0.0086
5,card_age_days,434.1018,432.3607,41254.2903,41692.4915,0.0085
35,share_spend_7d,0.0654,0.0639,0.0435,0.0418,0.0077


## Итоги

- Витрина `data/features.parquet`: 200 039 строк, одна на клиента эксперимента, 46 признаков, собирается за
  десяток секунд.
- Признаки строятся только из данных до `CUTOFF` - это гарантирует фильтр по дате, проверено пересчётом шести
  признаков из сырых строк. Target и treatment в признаках не участвуют, поэтому утечки между частями разбиения нет.
- Баланс по группам (|SMD| не больше 0,009 по 45 числовым признакам, пол - на этапе 2) с этим согласуется, но сам
  по себе не доказывает отсутствие утечки: признак после коммуникации, на который она не повлияла, тоже был бы
  сбалансирован.
- NULL только там, где значение не определено. Почти постоянных признаков (ненулевых меньше 0,5%) в витрине нет.